# Stage 2 Kaggle Smoke Notebook

Smoke-only workflow for Stage 2 on Kaggle/Linux. It keeps the attach-only data contract explicit, uses the canonical latent builder and smoke trainer, and writes artifacts under /kaggle/working.

Recovery precedence: latest_step.ckpt -> interrupt.ckpt -> latest.ckpt -> best.ckpt.
Export/evaluation preference: best.ckpt -> latest.ckpt.

In [ ]:
# Parameters
HARDWARE_PROFILE = "p100"  # p100 | t4x2
RUN_MODE = "first"         # first | resume
DATASET_SLUG = "balraj98/modelnet40-princeton-3d-object-dataset"
STAGE1_CHECKPOINT = "/kaggle/working/checkpoints/best.ckpt"
RESUME_CHECKPOINT_PATH = ""
LATENT_SCHEMA_VERSION = "stage2-latent-v1"
LATENT_PREP_CHECKPOINT_PREFERENCE = "latest_step,interrupt,latest,best"
SMOKE_NUM_SAMPLES = 4
SMOKE_BATCH_SIZE = 8

In [ ]:
import os
from pathlib import Path

ROOT = Path('/kaggle/working/DoAn_backup')
if not ROOT.exists():
    ROOT = Path.cwd()
os.chdir(ROOT)

hardware_map = {'p100': 'configs/hardware_p100.yaml', 't4x2': 'configs/hardware_t4x2.yaml'}
if HARDWARE_PROFILE not in hardware_map:
    raise ValueError('HARDWARE_PROFILE must be p100 or t4x2')
if RUN_MODE not in {'first', 'resume'}:
    raise ValueError('RUN_MODE must be first or resume')

dataset_name = DATASET_SLUG.split('/')[-1]
os.environ['DATASET_SLUG'] = DATASET_SLUG
os.environ['DATASET_ROOT'] = f'/kaggle/input/{dataset_name}'
os.environ['OUTPUT_ROOT'] = '/kaggle/working'
os.environ['TRAIN_CONFIG'] = 'configs/train_stage2.yaml'
os.environ['DATA_CONFIG'] = 'configs/data_stage2.yaml'
os.environ['HARDWARE_CONFIG'] = hardware_map[HARDWARE_PROFILE]
os.environ['STAGE1_CHECKPOINT_PATH'] = STAGE1_CHECKPOINT
if RUN_MODE == 'resume':
    os.environ['RESUME_CHECKPOINT_PATH'] = RESUME_CHECKPOINT_PATH or '/kaggle/working/checkpoints/latest_step.ckpt'

print('Project root:', ROOT)
print('DATASET_ROOT:', os.environ['DATASET_ROOT'])
print('TRAIN_CONFIG:', os.environ['TRAIN_CONFIG'])
print('HARDWARE_CONFIG:', os.environ['HARDWARE_CONFIG'])
print('RUN_MODE:', RUN_MODE)

## Bootstrap and environment checks

This validates Kaggle-compatible runtime conditions and prepares deterministic package install order. Outputs and metadata are written to /kaggle/working only.

In [ ]:
!bash scripts/kaggle_bootstrap.sh

In [ ]:
from pathlib import Path
dataset_root = Path(os.environ['DATASET_ROOT'])
output_root = Path(os.environ['OUTPUT_ROOT'])
stage1_ckpt = Path(os.environ['STAGE1_CHECKPOINT_PATH'])
print(f'Dataset root exists: {dataset_root.exists()} -> {dataset_root}')
print(f'Output root exists:  {output_root.exists()} -> {output_root}')
print(f'Stage 1 checkpoint candidate: {stage1_ckpt} (exists={stage1_ckpt.exists()})')
if not dataset_root.exists():
    raise RuntimeError('Dataset root missing. Attach the dataset in Kaggle UI before proceeding.')

## Latent manifest prep and validation

Stage 2 smoke training consumes manifest-backed latents. If the manifest tree is missing or stale, rebuild it with the canonical latent dataset builder.

Resume precedence for recovery stays explicit: latest_step -> interrupt -> latest -> best.

In [ ]:
import subprocess
from pathlib import Path
build_cmd = [
    'python', 'scripts/build_latent_dataset.py',
    '--train-config', 'configs/train_stage1.yaml',
    '--data-config', 'configs/data_stage2.yaml',
    '--dataset-root', os.environ['DATASET_ROOT'],
    '--output-root', os.environ['OUTPUT_ROOT'],
    '--checkpoint-path', os.environ['STAGE1_CHECKPOINT_PATH'],
    '--checkpoint-preference', LATENT_PREP_CHECKPOINT_PREFERENCE,
    '--split', 'both',
    '--batch-size', str(SMOKE_BATCH_SIZE),
    '--latent-schema-version', LATENT_SCHEMA_VERSION,
    '--enforce-kaggle',
]
print(' '.join(build_cmd))
subprocess.run(build_cmd, check=True)
manifest_root = Path('/kaggle/working/cache/stage2_latents') / LATENT_SCHEMA_VERSION / 'manifests'
for name in ['latent_manifest_train.jsonl', 'latent_manifest_test.jsonl']:
    path = manifest_root / name
    print(f'{path} exists={path.exists()}')

In [ ]:
import json
from pathlib import Path
manifest_root = Path('/kaggle/working/cache/stage2_latents') / LATENT_SCHEMA_VERSION / 'manifests'
for manifest_name in ['latent_manifest_train.jsonl', 'latent_manifest_test.jsonl']:
    manifest_path = manifest_root / manifest_name
    if not manifest_path.exists():
        raise RuntimeError(f'Missing latent manifest: {manifest_path}')
    first_line = manifest_path.read_text(encoding='utf-8').splitlines()[0]
    sample = json.loads(first_line)
    print(f"{manifest_name}: first sample_uid={sample.get('sample_uid')} token_shape={sample.get('token_shape')} checkpoint={sample.get('checkpoint_path')}")

## Stage 2 smoke training

Use the canonical smoke trainer only. This notebook stays smoke-scoped and does not expand into full Stage 2 training.

Recovery precedence: latest_step.ckpt -> interrupt.ckpt -> latest.ckpt -> best.ckpt.
Export/evaluation preference: best.ckpt -> latest.ckpt.

In [ ]:
smoke_cmd = [
    'python', 'scripts/train_stage2.py',
    '--config', os.environ['TRAIN_CONFIG'],
    '--hardware', os.environ['HARDWARE_CONFIG'],
    '--data-config', os.environ['DATA_CONFIG'],
    '--output-root', os.environ['OUTPUT_ROOT'],
    '--stage1-checkpoint', os.environ['STAGE1_CHECKPOINT_PATH'],
    '--run-id', 'kaggle-stage2-smoke',
    '--device', 'auto',
]
if RUN_MODE == 'resume':
    smoke_cmd += ['--resume-from', os.environ.get('RESUME_CHECKPOINT_PATH', '/kaggle/working/checkpoints/latest_step.ckpt')]
print(' '.join(smoke_cmd))
subprocess.run(smoke_cmd, check=True)

In [ ]:
from pathlib import Path
checkpoint_dir = Path('/kaggle/working/checkpoints')
print('Resume precedence: latest_step -> interrupt -> latest -> best')
for name in ['latest_step.ckpt', 'interrupt.ckpt', 'latest.ckpt', 'best.ckpt']:
    path = checkpoint_dir / name
    print(f'  {name}: exists={path.exists()} path={path}')
print('Export/evaluation preference: best -> latest')
for name in ['best.ckpt', 'latest.ckpt']:
    path = checkpoint_dir / name
    print(f'  {name}: exists={path.exists()} path={path}')

## Checkpoint integrity and smoke report inspection

The smoke trainer writes run metadata, smoke metrics, summary JSON, and checkpoint integrity reports under `/kaggle/working/logs` and `/kaggle/working/runs`. Inspect these artifacts after training to confirm the manifest-backed smoke run and recovery state.

In [ ]:
import json
from pathlib import Path
log_dir = Path('/kaggle/working/logs')
summary_path = log_dir / 'stage2_smoke_summary.json'
integrity_path = log_dir / 'stage2_checkpoint_integrity.json'
metrics_path = log_dir / 'stage2_smoke_metrics.jsonl'
print(f'Summary exists:   {summary_path.exists()} -> {summary_path}')
print(f'Integrity exists:  {integrity_path.exists()} -> {integrity_path}')
print(f'Metrics exists:    {metrics_path.exists()} -> {metrics_path}')
if summary_path.exists():
    summary = json.loads(summary_path.read_text(encoding='utf-8'))
    print('\nSmoke summary keys:')
    print(sorted(summary.keys()))
    print('\nSmoke run summary excerpt:')
    print(json.dumps({k: summary.get(k) for k in ['resume_path', 'smoke_report', 'metadata_path', 'checkpoint_dir', 'best_ckpt_path', 'latest_ckpt_path']}, indent=2))
if integrity_path.exists():
    integrity = json.loads(integrity_path.read_text(encoding='utf-8'))
    print('\nCheckpoint integrity excerpt:')
    print(json.dumps(integrity, indent=2)[:2000])

## Artifact listing and export guidance

Expected Kaggle-friendly outputs:
- `/kaggle/working/checkpoints`
- `/kaggle/working/logs`
- `/kaggle/working/cache/stage2_latents/stage2-latent-v1`
- `/kaggle/working/runs/<run_id>/metadata`

Use `best.ckpt` for export/evaluation and `latest_step.ckpt` for recovery messaging.

In [ ]:
from pathlib import Path
targets = [
    Path('/kaggle/working/checkpoints'),
    Path('/kaggle/working/logs'),
    Path('/kaggle/working/cache/stage2_latents') / LATENT_SCHEMA_VERSION,
    Path('/kaggle/working/runs'),
]
for target in targets:
    print(f'\n[{target}] exists={target.exists()}')
    if target.exists():
        for path in sorted(target.rglob('*'))[:50]:
            if path.is_file():
                print('  ', path)